# FlashEats — Class 8
## Building a Simple, Dependable Data Pipeline

### The story

Yesterday, our FlashEats workflow model worked.

We could calculate:

- Late Delivery Rate
- Order-to-Pickup Time
- Transit Time
- Customer interaction signals
- Intervention metrics

Today the client asks:

> **“Great. Can this run every morning without you?”**

That changes the engineering problem.

A successful notebook run is not the same thing as a dependable pipeline.

Today's goal:

```text
ONE COMMAND
    ↓
EXTRACT
    ↓
VALIDATE
    ↓
CLEAN
    ↓
TRANSFORM
    ↓
SAVE
    ↓
LOG
```

and if something is unsafe:

```text
FAIL CLEARLY
+
DO NOT PUBLISH BAD OUTPUT
```

In [1]:
!pip -q install pandas requests flask

In [2]:
# SETUP — local first, Colab upload only if necessary.
# No matplotlib is required for this walkthrough.

import json
import os
import subprocess
import sys
import time
import zipfile
from pathlib import Path

import pandas as pd


def find_pack_root():
    cwd = Path.cwd()

    # Fast path: notebook is already inside the pack.
    if (cwd / "database" / "flasheats.db").exists():
        return cwd

    # Common folder names.
    for name in [
        "FlashEats_Classroom_Pack_V2",
        "FlashEats_Classroom_Pack",
    ]:
        candidate = cwd / name
        if (candidate / "database" / "flasheats.db").exists():
            return candidate

    # Search locally, then Colab.
    for root in [cwd, cwd.parent, Path("/content")]:
        if not root.exists():
            continue

        for candidate in root.rglob("*"):
            if (
                candidate.is_dir()
                and (candidate / "database" / "flasheats.db").exists()
            ):
                return candidate

    return None


BASE = find_pack_root()

if BASE is None:
    try:
        from google.colab import files

        print("Upload FlashEats_Class8_Classroom_Pack.zip")
        uploaded = files.upload()
        zip_name = next(
            name for name in uploaded
            if name.endswith(".zip")
        )

        extract_dir = Path("/content/flasheats_class8")
        extract_dir.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(zip_name, "r") as z:
            z.extractall(extract_dir)

        BASE = find_pack_root()

    except Exception as exc:
        print("Automatic setup failed:", exc)

if BASE is None:
    raise FileNotFoundError(
        "Could not locate a FlashEats pack containing database/flasheats.db"
    )

PROJECT = BASE / "Class8_Project"

print("Pack:", BASE)
print("Project:", PROJECT)
print("Database exists:", (BASE / "database" / "flasheats.db").exists())
print("run_pipeline.py exists:", (PROJECT / "run_pipeline.py").exists())

Pack: /Users/lingaraghavendra/Desktop/fde_assignment/Version_3/exercises/FlashEats_Classroom_Pack_V2
Project: /Users/lingaraghavendra/Desktop/fde_assignment/Version_3/exercises/FlashEats_Classroom_Pack_V2/Class8_Project
Database exists: True
run_pipeline.py exists: True


# 1. Working once ≠ dependable

Imagine yesterday's notebook contains:

```python
orders = load_orders()
dispatch = fetch_dispatch()
journey = build_journey(orders, dispatch)
journey.to_csv("order_journey.csv")
```

It works today.

## What assumptions are hidden?


1. What if the API returns only page 1?
2. What if the API returns HTTP 500?
3. What if `promised_eta` disappears?
4. What if the source is stale?
5. What if the job is run twice?
6. What if it crashes halfway through?

### Deduction

> **Dependability means making important assumptions explicit and testable.**

# 2. Turn the notebook into stages

We will use five explicit stages:

```text
EXTRACT → VALIDATE → CLEAN → TRANSFORM → SAVE
```

Why stages?

Because when something fails, we want to know **what responsibility failed**.

Open the actual project structure:

In [3]:
for path in sorted(PROJECT.rglob("*")):
    if path.is_file():
        print(path.relative_to(PROJECT))

GATE2_DATA_READINESS.md
GATE2_DATA_READINESS_TEMPLATE.md
README.md
TROUBLESHOOTING.md
config/.env.example
pipeline/__init__.py
pipeline/clean.py
pipeline/config.py
pipeline/extract.py
pipeline/logging_utils.py
pipeline/save.py
pipeline/transform.py
pipeline/validate.py
requirements.txt
run_pipeline.py


### Discussion

- Why should validation happen before transformation?
- Why preserve raw API responses?
- Why is `save()` a separate stage?
- Would you put every operation in one giant function?

### Deduction

> Functions are not the goal. **Isolation of responsibility** is the goal.

**My answers to the discussion**

**Why validate before transforming?** The transform trusts its input. In `transform.py` the delay is just `actual_delivery_at - promised_eta`, so if a column is missing or full of junk you either get a crash in the middle of the merges or, worse, a table that looks fine and has wrong numbers. In Class 6 we found these problems by hand; checking them first means bad input never reaches the output.

**Why keep the raw API responses?** They are the only proof of what dispatch told us on that day. The API is live, so calling it again next week can give different ETAs or drivers. With the raw pages I can rebuild the output or explain a strange number later without asking the dispatch team.

**Why is `save()` separate?** It is the one step with side effects that other people see. Keeping it last and on its own means nothing is published unless every stage before it worked, and the "replace the partition" logic lives in one place.

**One giant function?** I wouldn't. My Class 5 notebook was close to that, and when something broke I had to rerun everything to find which part. Separate stages give separate log lines and separate places to look.

# 3. Configuration should not be buried in code

Bad:

```python
API_URL = "http://localhost:8000"
MAX_RETRIES = 3
OUTPUT = "/Users/me/Desktop/output.csv"
```

Better:

```text
Code = behavior
Environment/config = where and how it runs
```

Let's inspect the example configuration.

In [4]:
print((PROJECT / "config" / ".env.example").read_text())

# FlashEats Class 8 pipeline configuration

DISPATCH_API_URL=http://127.0.0.1:8000
MAX_RETRIES=3
RETRY_BASE_SECONDS=1
MAX_DATA_AGE_DAYS=60
PAGE_SIZE=200
START_MOCK_API=true
LOG_LEVEL=INFO



### Discussion

1. Which values could differ between local, staging and production?
2. Which values should never be committed if they are secrets?
3. Why is `MAX_RETRIES` configuration but retry logic code?

### Deduction

> A pipeline is easier to move between environments when deployment-specific values are outside core business logic.

**My answers**

1. The API address (`DISPATCH_API_URL`) changes for sure. `START_MOCK_API` should only be true on a laptop. `LOG_LEVEL` could be DEBUG locally and INFO in production. `MAX_DATA_AGE_DAYS` depends on how often the source is refreshed in that environment. Page size and retry settings might be tuned to each environment's API limits.
2. Anything that is a credential: API tokens, database passwords, cloud keys. The current example file has none of those, which is right, and the real `.env` should be gitignored.
3. `MAX_RETRIES` is a setting someone might want to change at 2am without deploying code. The logic of which errors are retryable and how the wait grows is a decision that needs review, because a mistake there (for example retrying a 400 forever) is a bug, not a setting. Later in this notebook I use `MAX_DATA_AGE_DAYS` to show how changing a setting changes the result without touching the code.

# 4. Validation becomes a gate

Class 6 taught us how to discover data-quality rules.

Class 8 changes the question:

> **How do we make those rules execute every run?**

Our pipeline checks:

- required columns,
- duplicate order IDs,
- critical missing completion timestamps,
- freshness,
- required Dispatch fields.

Let's inspect the validation module.

In [5]:
validation_code = (PROJECT / "pipeline" / "validate.py").read_text()

print(validation_code[:5000])

from dataclasses import dataclass
from datetime import date
import pandas as pd


class ValidationError(ValueError):
    pass


@dataclass
class CheckResult:
    check: str
    status: str
    detail: str


ORDER_REQUIRED_COLUMNS = {
    "order_id",
    "customer_id",
    "restaurant_id",
    "created_at",
    "promised_eta",
    "pickup_at",
    "actual_delivery_at",
    "final_status",
}


def validate_required_columns(df, required_columns, dataset_name):
    missing = sorted(set(required_columns) - set(df.columns))

    if missing:
        raise ValidationError(
            f"{dataset_name}: missing required columns: {missing}"
        )

    return CheckResult(
        check=f"{dataset_name}.required_columns",
        status="PASS",
        detail=f"{len(required_columns)} required columns present",
    )


def validate_order_uniqueness(orders):
    duplicate_rows = int(orders.duplicated("order_id", keep=False).sum())

    # This source is intentionally known to contain a few dupli

### Story

Suppose `promised_eta` disappears tomorrow.

A weak pipeline continues until some later line crashes — or worse, produces a misleading output.

A dependable pipeline should say:

```text
VALIDATION FAILED
orders: missing required columns: ['promised_eta']

No processed output written.
```

### Deduction

> **Validation protects downstream consumers from silently bad output.**

# 5. Retries: retry the right failures

The FlashEats Dispatch API deliberately behaves like a real external dependency:

- page 3 fails once with HTTP 500,
- page 5 rate-limits once with HTTP 429,
- pagination must be completed.

A retry is appropriate for a **transient** failure.

A retry is usually not useful for:

- missing required column,
- invalid credentials,
- incompatible schema,
- a business-rule failure.

Let's inspect the retry logic.

In [6]:
extract_code = (PROJECT / "pipeline" / "extract.py").read_text()

start = extract_code.find("def _request_page")
end = extract_code.find("def extract_dispatch_api")

print(extract_code[start:end])

def _request_page(session, url, page, page_size, max_retries, base_seconds, logger):
    for attempt in range(1, max_retries + 1):
        try:
            response = session.get(
                f"{url}/dispatch/orders",
                params={"page": page, "page_size": page_size},
                timeout=5,
            )

            if response.status_code == 200:
                return response.json()

            if response.status_code in {429, 500, 502, 503, 504}:
                retry_after = response.headers.get("Retry-After")
                if response.status_code == 429:
                    try:
                        body = response.json()
                        retry_after = body.get(
                            "retry_after_seconds",
                            retry_after,
                        )
                    except Exception:
                        pass

                wait_seconds = (
                    float(retry_after)
                    if retry_af

### Discussion

1. Why limit the number of retries?
2. Why wait longer between repeated attempts?
3. Why should HTTP 429 and HTTP 500 be treated differently from a schema failure?
4. Could retrying a write operation be dangerous?

### Deduction

> **Retry transient failures; fail fast on conditions that require human or code changes.**

**My answers**

1. A retry limit turns "the job is hanging" into "the job failed with a message". Without it a dead API means a job that never ends and never alerts anyone, and it keeps adding load to a service that may already be struggling.
2. The first retry is cheap and often enough. If it fails again, the problem is probably bigger, so waiting longer (1s, then 2s here) gives the other side time to recover. For 429 the code uses the server's `retry_after_seconds`, which is the polite way to handle rate limits.
3. A 500 or 429 says "not right now", and a later call can work. A missing column says "this data is wrong", and it will be wrong on every retry. Retrying it only delays the failure and makes the log harder to read.
4. Yes. If a write succeeded but the response got lost, a retry writes it again: two refunds, two rows, two notifications. You can only retry writes safely if the write is idempotent, like this pipeline's "replace this partition" save.

# 6. Idempotency: what happens if we run twice?

A job that appends the same 1,600 orders every time is repeatable — but not dependable.

For this class:

```text
same logical run_date
        ↓
same output partition
        ↓
atomically replace output
```

Let's inspect the save stage.

In [7]:
save_code = (PROJECT / "pipeline" / "save.py").read_text()
print(save_code)

from pathlib import Path
import json
import os
import tempfile


def atomic_write_csv(df, destination: Path):
    destination.parent.mkdir(parents=True, exist_ok=True)

    with tempfile.NamedTemporaryFile(
        mode="w",
        suffix=".csv",
        delete=False,
        dir=destination.parent,
    ) as tmp:
        temp_path = Path(tmp.name)

    try:
        df.to_csv(temp_path, index=False)
        os.replace(temp_path, destination)
    finally:
        if temp_path.exists():
            temp_path.unlink(missing_ok=True)


def save_outputs(
    journey,
    metrics,
    validation_results,
    project_root: Path,
    run_date: str,
    logger,
):
    partition = (
        project_root
        / "data"
        / "processed"
        / f"run_date={run_date}"
    )

    partition.mkdir(parents=True, exist_ok=True)

    journey_path = partition / "order_journey.csv"
    metrics_path = partition / "metrics.json"
    validation_path = partition / "validation_report.json"

    # Idemp

### Discussion

1. Why partition outputs by logical run date?
2. Why replace instead of append for this pipeline?
3. When would append actually be correct?
4. Why write a temporary file before replacing the final file?

### Deduction

> **A rerun should not create unintended duplicate effects.**

**My answers**

1. Partitioning by logical run date ties each output to the business day it describes, not the time the job happened to run. If the 2026-09-22 job is late or rerun, it still lands in `run_date=2026-09-22` and nothing else is touched.
2. Every run rebuilds the full journey from the full source, so the correct output is always the latest snapshot for that day. Appending would stack copies and double every count.
3. Append is right when the input really is new each time and has a unique key, like new events since the last run, or a history table of pipeline runs. Even then I'd want a dedup on the key in case a run is repeated.
4. Writing directly to the final file means a crash halfway leaves a broken CSV that looks like real output. With a temp file plus `os.replace`, the switch happens in one step, so the final path always has either the old complete file or the new one.

# 7. Logging should help the next engineer

Bad log:

```text
ERROR
```

Useful log:

```text
Dispatch API failure
page=3
status=500
attempt=2/3
```

A log should help answer:

- which stage?
- which source?
- how many records?
- which attempt?
- what failed?
- did output get published?

The pipeline writes both console and file logs.

In [8]:
print((PROJECT / "pipeline" / "logging_utils.py").read_text())

import logging
from pathlib import Path


def build_logger(log_path: Path, level: str = "INFO") -> logging.Logger:
    log_path.parent.mkdir(parents=True, exist_ok=True)

    logger = logging.getLogger("flasheats_pipeline")
    logger.setLevel(getattr(logging, level, logging.INFO))
    logger.handlers.clear()

    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s"
    )

    console = logging.StreamHandler()
    console.setFormatter(formatter)

    file_handler = logging.FileHandler(log_path)
    file_handler.setFormatter(formatter)

    logger.addHandler(console)
    logger.addHandler(file_handler)

    return logger



### Deduction

> **A dependable pipeline is diagnosable, not merely executable.**

# 8. Run the complete flow with one command

This is the handoff moment.

From `Class8_Project`:

```bash
python run_pipeline.py --run-date 2026-09-22
```

The command should:

1. start the mock API if needed,
2. extract local sources,
3. retrieve every Dispatch API page,
4. retry transient failures,
5. preserve raw pages,
6. validate inputs,
7. clean the agreed business grain,
8. build the Class 7 order journey,
9. save metrics + validation report,
10. write logs.

Run it from this notebook:

In [9]:
# Install only the small dependencies used by the project.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT / "requirements.txt")],
    check=True,
)

result = subprocess.run(
    [
        sys.executable,
        str(PROJECT / "run_pipeline.py"),
        "--run-date",
        "2026-09-22",
    ],
    cwd=PROJECT,
    text=True,
    capture_output=True,
)

print("RETURN CODE:", result.returncode)
print("\nSTDOUT:\n", result.stdout[-5000:])
print("\nSTDERR:\n", result.stderr[-3000:])

RETURN CODE: 0

STDOUT:
 
PIPELINE SUCCESS
{
  "orders_in_journey": 1600,
  "measurable_deliveries": 1495,
  "late_delivery_rate_pct": 56.39,
  "median_order_to_pickup_min": 26.07,
  "median_transit_min": 47.23,
  "orders_with_intervention": 430
}

Outputs:
  journey: /Users/lingaraghavendra/Desktop/fde_assignment/Version_3/exercises/FlashEats_Classroom_Pack_V2/Class8_Project/data/processed/run_date=2026-09-22/order_journey.csv
  metrics: /Users/lingaraghavendra/Desktop/fde_assignment/Version_3/exercises/FlashEats_Classroom_Pack_V2/Class8_Project/data/processed/run_date=2026-09-22/metrics.json
  validation: /Users/lingaraghavendra/Desktop/fde_assignment/Version_3/exercises/FlashEats_Classroom_Pack_V2/Class8_Project/data/processed/run_date=2026-09-22/validation_report.json


STDERR:
 2026-09-25 13:52:13,461 | INFO | Pipeline started | run_date=2026-09-22 chaos=none
2026-09-25 13:52:14,473 | INFO | Starting local mock dispatch API
2026-09-25 13:52:15,000 | INFO | Mock dispatch API is hea

# 9. Prove idempotency

Run exactly the same logical job again.

What should happen?

- output row count should not double,
- the same partition should be replaced,
- the final business dataset should remain one row per order.

In [10]:
output_path = (
    PROJECT
    / "data"
    / "processed"
    / "run_date=2026-09-22"
    / "order_journey.csv"
)

first = pd.read_csv(output_path)
first_rows = len(first)

second_run = subprocess.run(
    [
        sys.executable,
        str(PROJECT / "run_pipeline.py"),
        "--run-date",
        "2026-09-22",
    ],
    cwd=PROJECT,
    text=True,
    capture_output=True,
)

second = pd.read_csv(output_path)
second_rows = len(second)

print("First run rows:", first_rows)
print("Second run rows:", second_rows)
print("Rows doubled?", second_rows == first_rows * 2)
print("Same logical row count?", first_rows == second_rows)

First run rows: 1600
Second run rows: 1600
Rows doubled? False
Same logical row count? True


In [11]:
# the log file is shared by every run with this run_date, so after two runs it should show two complete runs
log_path = PROJECT / "logs" / "pipeline_2026-09-22.log"
log_lines = log_path.read_text().splitlines()
print("log lines so far:", len(log_lines))
print("runs started:", sum("Pipeline started" in l for l in log_lines),
      "| completed:", sum("Pipeline completed" in l for l in log_lines))
print("retry warnings:", sum("Retryable dispatch API failure" in l for l in log_lines))

# file permissions of what was published
for p in sorted(output_path.parent.iterdir()):
    print(p.name, oct(p.stat().st_mode & 0o777))

log lines so far: 48
runs started: 2 | completed: 2
retry warnings: 4
metrics.json 0o644
order_journey.csv 0o600
validation_report.json 0o644


The rerun kept 1600 rows, and the log shows two full runs one after the other, each with the same two retries (page 3 and page 5). The log file is appended to, not replaced, so it keeps the history of every run for that date.

One small thing I found: `order_journey.csv` ends up with permissions `0o600` (only my user can read it), while the JSON files are `0o644`. That comes from `tempfile.NamedTemporaryFile`, which creates the temp file as private, and `os.replace` keeps those permissions. On a shared server another service or user might not be able to read the main output. Easy to fix with a `chmod`, but good to know before handing it over.

### Deduction

> **“Can rerun” and “safe to rerun” are different properties.**

# 10. Controlled failure: break the schema

A dependable pipeline should not just succeed nicely.

It should fail nicely.

Run:

In [12]:
failure = subprocess.run(
    [
        sys.executable,
        str(PROJECT / "run_pipeline.py"),
        "--run-date",
        "2026-09-22",
        "--chaos",
        "missing_column",
    ],
    cwd=PROJECT,
    text=True,
    capture_output=True,
)

print("RETURN CODE:", failure.returncode)
print(failure.stdout[-3000:])

RETURN CODE: 2

PIPELINE FAILED: orders: missing required columns: ['promised_eta']



### Discuss

1. Did the pipeline explain what was wrong?
2. Did it keep retrying a non-retryable problem?
3. Did it publish a new processed dataset?
4. What would an on-call engineer need next?

### Deduction

> **Dependability includes safe, understandable failure.**

In [13]:
# what does the log say about the failed run?
log_lines = log_path.read_text().splitlines()
last_start = max(i for i, l in enumerate(log_lines) if "Pipeline started" in l)
for line in log_lines[last_start:]:
    print(line.split(" | ", 1)[1][:180])

INFO | Pipeline started | run_date=2026-09-22 chaos=missing_column
INFO | Starting local mock dispatch API
INFO | Mock dispatch API is healthy
INFO | Extracted local sources | orders=1603 interactions=497 interventions=430
WARNING | CHAOS: dropping promised_eta to simulate schema break
INFO | Fetched dispatch API page | page=1 rows=200 cumulative=200
INFO | Fetched dispatch API page | page=2 rows=200 cumulative=400
WARNING | Retryable dispatch API failure | page=3 status=500 attempt=1/3 wait=1.0s
INFO | Fetched dispatch API page | page=3 rows=200 cumulative=600
INFO | Fetched dispatch API page | page=4 rows=200 cumulative=800
WARNING | Retryable dispatch API failure | page=5 status=429 attempt=1/3 wait=1.0s
INFO | Fetched dispatch API page | page=5 rows=200 cumulative=1000
INFO | Fetched dispatch API page | page=6 rows=200 cumulative=1200
INFO | Fetched dispatch API page | page=7 rows=200 cumulative=1400
INFO | Fetched dispatch API page | page=8 rows=200 cumulative=1600
INFO | Dispatch

**My answers**

1. Yes. The notebook output only shows `PIPELINE FAILED: orders: missing required columns: ['promised_eta']`, but that is already enough, and the log (above) has the full story: which chaos scenario, that dispatch extraction still finished (1600 records), and that it stopped at the validation gate.
2. No. There is exactly one ERROR line and no retry for the validation problem. The two WARNING retries in this run are the dispatch API ones (page 3 and 5), which are transient and happened before validation.
3. No. The ERROR line ends with "no processed output written", and the partition still has the files from the second successful run.
4. The log path and run date (easy here since the log is named by date), who owns the orders schema and what changed, whether the old partition is still OK to show to users, and a way to be told about it. Right now nothing alerts anyone, someone has to look at the exit code or the log.

### More failure tests

`run_pipeline.py` has two more chaos scenarios (`duplicate_order`, `stale_data`). I also tried a config change instead of a code change: setting `MAX_DATA_AGE_DAYS=20` on the normal data, which is 25 days old, so the freshness check should fail. At the end I run the normal job once more so the saved partition comes from a clean run.

In [14]:
def run_pipeline(*args, env_extra=None):
    env = dict(os.environ, **(env_extra or {}))
    return subprocess.run(
        [sys.executable, str(PROJECT / "run_pipeline.py"), "--run-date", "2026-09-22", *args],
        cwd=PROJECT,
        text=True,
        capture_output=True,
        env=env,
    )

tests = {
    "duplicate_order": run_pipeline("--chaos", "duplicate_order"),
    "stale_data": run_pipeline("--chaos", "stale_data"),
    "MAX_DATA_AGE_DAYS=20": run_pipeline(env_extra={"MAX_DATA_AGE_DAYS": "20"}),
}
for name, r in tests.items():
    print(f"{name:22s} exit={r.returncode} | {r.stdout.strip().splitlines()[0]}")

dup_lines = [l.split(" | ", 2)[-1] for l in tests["duplicate_order"].stderr.splitlines() if "uniqueness" in l or "Cleaned" in l]
print("\nduplicate_order details:", dup_lines)

restore = run_pipeline()
print("\nclean run to restore the partition, exit code:", restore.returncode)

duplicate_order        exit=0 | PIPELINE SUCCESS
stale_data             exit=2 | PIPELINE FAILED: orders data is stale: latest_created_at=2025-08-28 23:33:00, age_days=390, allowed=60
MAX_DATA_AGE_DAYS=20   exit=2 | PIPELINE FAILED: orders data is stale: latest_created_at=2026-08-28 23:33:00, age_days=25, allowed=20

duplicate_order details: ['Validation | check=orders.order_id_uniqueness status=WARN detail=duplicate rows involved=8', 'Cleaned orders | input_rows=1604 output_rows=1600 deduplicated=4']



clean run to restore the partition, exit code: 0


In [15]:
# one row per run, built only from the log file
runs = []
for line in log_path.read_text().splitlines():
    parts = line.split(" | ", 2)
    if len(parts) < 3:
        continue
    ts, level, msg = parts
    if msg.startswith("Pipeline started"):
        runs.append({"started": ts[11:19], "chaos": msg.split("chaos=")[1], "warnings": 0, "errors": 0, "result": "?"})
    elif runs:
        runs[-1]["warnings"] += level == "WARNING"
        runs[-1]["errors"] += level == "ERROR"
        if msg.startswith("Pipeline completed"):
            runs[-1]["result"] = "success"
        elif msg.startswith("Pipeline stopped"):
            runs[-1]["result"] = "stopped: " + msg.split("error=")[1].split(" | ")[0][:70]
print(pd.DataFrame(runs).to_string(index=False))

 started           chaos  warnings  errors                                                                          result
13:52:13            none         2       0                                                                         success
13:52:17            none         2       0                                                                         success
13:52:21  missing_column         3       1                     stopped: orders: missing required columns: ['promised_eta']
13:52:25 duplicate_order         3       0                                                                         success
13:52:29      stale_data         3       1 stopped: orders data is stale: latest_created_at=2025-08-28 23:33:00, age_days=
13:52:33            none         2       1 stopped: orders data is stale: latest_created_at=2026-08-28 23:33:00, age_days=
13:52:37            none         2       0                                                                         success


Reading only the log, I can rebuild all seven runs of the day: 4 successes and 3 stops at the validation gate.

- `duplicate_order` succeeded with one extra WARNING (the chaos message). The uniqueness check went from 6 to 8 rows involved and clean removed 4 duplicates, so the output was still 1600 orders.
- `stale_data` stopped at freshness (`age_days=390, allowed=60`).
- The `MAX_DATA_AGE_DAYS=20` run also stopped at freshness (`age_days=25, allowed=20`) without any code change, which shows the setting really lives outside the code.

What I'd improve: that config run shows up in the log as `chaos=none` and looks like a normal run that failed. The log never records the settings used, so an on-call engineer could not tell that someone changed `MAX_DATA_AGE_DAYS`. I'd log the config values at the start of each run.

# 11. Gate 2 — Data readiness review

The final output of this module is not merely `pipeline.py`.

We need to be able to answer:

> **Is the data foundation ready for the next product/AI phase?**

Review the template:

In [16]:
print((PROJECT / "GATE2_DATA_READINESS_TEMPLATE.md").read_text())

# Gate 2 — Data Readiness Review

## Pipeline run

- [ ] Complete flow runs with one command
- [ ] Same run can be safely repeated
- [ ] Raw API responses are preserved
- [ ] Processed output is written only after validation

## Validation

| Check | Status | Evidence / note |
|---|---|---|
| Required columns |  |  |
| Critical nulls |  |  |
| Order uniqueness |  |  |
| Freshness |  |  |
| Dispatch retrieval completeness |  |  |

## Reliability

| Capability | Status | Evidence / note |
|---|---|---|
| Bounded retries |  |  |
| Useful failure message |  |  |
| Logging |  |  |
| Idempotent rerun |  |  |
| Configuration outside core logic |  |  |

## Known limitations

Document anything that remains unresolved.

Example:
- `driver_arrived_at_restaurant` is still not reliably captured.

## Gate decision

**READY / NOT READY**

Reason:



I filled the template in `Class8_Project/GATE2_DATA_READINESS.md` with the evidence from this notebook and the log. The decision part:

In [17]:
gate = (PROJECT / "GATE2_DATA_READINESS.md").read_text()
print(gate[gate.index("## Gate decision"):])

## Gate decision

**NOT READY**

Reason: the pipeline itself is dependable. It runs with one command, retries only what it should, stops bad data at the gate and reruns safely. But the data foundation is not ready for the next phase. The main driver of lateness (the wait before pickup) has no events to explain it, the pickup fields my Class 7 metrics need are dropped by the transform, and the KPI definition still has no owner. I'd pass it once `estimated_pickup_at` and the driver fields are carried into the order journey, the KPI owner signs off the definition, and there is at least an alert on failed runs.



# The story to carry out of Class 8

We started with:

```text
“Here is a notebook that works.”
```

We ended with:

```text
ONE COMMAND
   ↓
KNOWN CONFIGURATION
   ↓
BOUNDED RETRIES
   ↓
VALIDATION GATES
   ↓
EXPLICIT TRANSFORMS
   ↓
IDEMPOTENT OUTPUT
   ↓
USEFUL LOGS
   ↓
READINESS DECISION
```

The core lesson:

> **A dependable pipeline is not one that never fails. It produces trustworthy output when things are healthy, and fails clearly and safely when they are not.**

# Knowledge check

## Poll 1
The Dispatch API returns HTTP 500 once. What should the pipeline usually do?

A. Delete the input  
B. Retry a bounded number of times  
C. Remove validation  
D. Retry forever


---

## Poll 2
`promised_eta` disappears from the source schema. What should happen?

A. Retry the API  
B. Fill the column with NULL and publish  
C. Fail the validation gate with a useful message  
D. Ignore it because yesterday's run worked


---

## Poll 3
A pipeline run for the same `run_date` is executed twice. Which behavior best demonstrates idempotency?

A. Double the rows  
B. Create two identical permanent copies  
C. Replace the same logical output without duplicate effects  
D. Refuse every rerun


---

## Poll 4
Which is the best log message?

A. `ERROR`  
B. `Something broke`  
C. `Dispatch page=5 status=429 attempt=2/3 wait=1s`  
D. `Try again`


---

## Poll 5
Which Class 6 concept changes meaningfully in Class 8?

A. We stop validating data  
B. Data-quality rules become executable gates on every run  
C. We replace business rules with infrastructure  
D. We remove source ownership

**My answers**

- **Poll 1: B.** Retry a bounded number of times. One 500 is usually temporary, and the limit stops it from hanging if it isn't.
- **Poll 2: C.** Fail the validation gate with a useful message. Retrying can't bring a column back, and NULL-filling it would publish a late rate that is quietly wrong.
- **Poll 3: C.** Replace the same logical output. That is what the save stage does for `run_date=2026-09-22`, and the row count stayed at 1600.
- **Poll 4: C.** It has the page, the status, the attempt number and the wait, the same format as the WARNING lines in our log.
- **Poll 5: B.** The Class 6 checks become code that runs every time. The 37 missing delivery times and the 3 duplicate orders we found by hand in Class 6 now show up as WARN lines in every run.